# U1-03 · IA clásica, scikit-learn y TensorFlow con los pingüinos

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 1*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) las redes se entrenan mucho más deprisa.

> Datos: palmerpenguins, de Horst, Hill y Gorman (2020), CC0. Gorman, K. B., Williams, T. D. y Fraser, W. R. (2014). *PLoS ONE*, 9(3), e90081. https://doi.org/10.1371/journal.pone.0090081

In [ ]:
# Descarga los datos y módulos auxiliares (solo la primera vez)
import os, urllib.request
for f in ["penguins.csv"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve("https://machinelearning-formacion.github.io/ifcd107/recursos/datos/u1/" + f, f)

## 1.1 · ¿Qué es la inteligencia artificial?

`a1_reglas.py`

In [ ]:
import csv

def especie_por_reglas(aleta, pico):
    """Reglas escritas a mano por un «experto»."""
    if aleta >= 207:
        return "Gentoo"
    if pico >= 44:
        return "Chinstrap"
    return "Adelie"

aciertos = total = 0
fallos = {}
with open("penguins.csv") as f:
    for fila in csv.DictReader(f):
        if fila["flipper_length_mm"] == "NA":
            continue                       # dos pingüinos sin medir
        real = fila["species"]
        prev = especie_por_reglas(float(fila["flipper_length_mm"]),
                                  float(fila["bill_length_mm"]))
        total += 1
        aciertos += prev == real
        if prev != real:
            fallos[(real, prev)] = fallos.get((real, prev), 0) + 1
print(f"aciertos: {aciertos} de {total} ({aciertos / total:.1%})")
for (real, prev), n in sorted(fallos.items()):
    print(f"  {real} tomado por {prev}: {n}")

`a2_aprendida.py`

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

p = pd.read_csv("penguins.csv").dropna(
    subset=["flipper_length_mm", "bill_length_mm"])
X = p[["flipper_length_mm", "bill_length_mm"]]
y = p.species
X_e, X_p, y_e, y_p = train_test_split(X, y, test_size=0.3,
                                      random_state=0, stratify=y)
arbol = DecisionTreeClassifier(max_depth=2, random_state=0)
arbol.fit(X_e, y_e)                 # las reglas salen de los datos
print(export_text(arbol, feature_names=["aleta", "pico"],
                  decimals=1))
print(f"acierto con pingüinos nuevos: {arbol.score(X_p, y_p):.1%}")

`a3_busqueda.py`

In [ ]:
import heapq
from collections import deque

META = (1, 2, 3, 4, 5, 6, 7, 8, 0)       # 0 = hueco
def vecinos(e):
    i = e.index(0); f, c = divmod(i, 3)
    for df, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        if 0 <= f + df < 3 and 0 <= c + dc < 3:
            j = (f + df) * 3 + c + dc
            l = list(e); l[i], l[j] = l[j], l[i]
            yield tuple(l)

def anchura(inicio):                     # búsqueda a ciegas
    vistos, cola = {inicio: 0}, deque([inicio])
    while cola:
        e = cola.popleft()
        if e == META:
            return vistos[e], len(vistos)
        for v in vecinos(e):
            if v not in vistos:
                vistos[v] = vistos[e] + 1; cola.append(v)

def manhattan(e):                        # heurística admisible
    return sum(abs(i // 3 - (v - 1) // 3) + abs(i % 3 - (v - 1) % 3)
               for i, v in enumerate(e) if v)

def a_estrella(inicio):                  # búsqueda informada
    g, cerrados = {inicio: 0}, set()
    abiertos = [(manhattan(inicio), inicio)]  # (g + h, estado)
    while abiertos:
        _, e = heapq.heappop(abiertos)
        if e == META:
            return g[e], len(cerrados)
        if e in cerrados:
            continue
        cerrados.add(e)
        for v in vecinos(e):
            if g[e] + 1 < g.get(v, 99):
                g[v] = g[e] + 1
                heapq.heappush(abiertos, (g[v] + manhattan(v), v))

inicio = (8, 6, 7, 2, 5, 4, 3, 0, 1)
print("anchura: %d movimientos, %d estados visitados"
      % anchura(inicio))
print("A*:      %d movimientos, %d estados expandidos"
      % a_estrella(inicio))

`a5_minimax.py`

In [ ]:
from functools import lru_cache

LINEAS = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7),
          (2, 5, 8), (0, 4, 8), (2, 4, 6)]
def ganador(t):
    for a, b, c in LINEAS:
        if t[a] != "." and t[a] == t[b] == t[c]:
            return t[a]
    return None

@lru_cache(maxsize=None)            # recuerda posiciones vistas
def minimax(t, turno):
    """Valor de la posición para X: 1 gana, 0 tablas, -1 pierde."""
    g = ganador(t)
    if g:
        return 1 if g == "X" else -1
    if "." not in t:
        return 0
    hijos = [minimax(t[:i] + turno + t[i + 1:],
                     "O" if turno == "X" else "X")
             for i, c in enumerate(t) if c == "."]
    return max(hijos) if turno == "X" else min(hijos)

print("valor del tablero vacío:", minimax("." * 9, "X"))
print("posiciones distintas evaluadas:",
      minimax.cache_info().currsize)
t = "XX.OO...."                          # juega X: ¿dónde?
for i in [i for i, c in enumerate(t) if c == "."]:
    print(f"X en la casilla {i}: valor",
          minimax(t[:i] + "X" + t[i + 1:], "O"))

`a4_tipos.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

p = pd.read_csv("penguins.csv").dropna()
X = StandardScaler().fit_transform(
    p[["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]])
# 1) supervisado: aprende con la especie como respuesta
sup = LogisticRegression().fit(X, p.species)
print(f"supervisado: acierto {sup.score(X, p.species):.3f}")
# 2) no supervisado: agrupa sin ver la especie
g = KMeans(3, n_init=10, random_state=0).fit_predict(X)
print("no supervisado, especies por grupo:")
print(pd.crosstab(g, p.species, rownames=["grupo"],
                  colnames=[""]).to_string())
# 3) por refuerzo: aprende probando, de la recompensa
rng = np.random.default_rng(0)
premio = [0.3, 0.5, 0.7]                 # desconocido para el agente
valor, veces = np.zeros(3), np.zeros(3)
for t in range(2000):
    a = rng.integers(3) if rng.random() < 0.1 else valor.argmax()
    r = rng.random() < premio[a]         # 1 si hay recompensa
    veces[a] += 1; valor[a] += (r - valor[a]) / veces[a]
print("refuerzo: estimaciones", valor.round(2),
      "| veces elegida", veces.astype(int))

## 1.6 · Las bibliotecas de la IA en Python

`e1_pandas.py`

In [ ]:
import pandas as pd

p = pd.read_csv("penguins.csv")
print(p.shape, "|", ", ".join(p.columns[:4]), "...")
grandes = p.loc[p.body_mass_g > 5500, ["species", "sex",
                                         "body_mass_g"]]
print("más de 5,5 kg:", len(grandes), grandes.sex.value_counts()
      .to_dict())
p["masa_kg"] = p.body_mass_g / 1000                 # columna nueva
p["pico_ratio"] = p.bill_length_mm / p.bill_depth_mm
tabla = p.pivot_table(index="species", columns="sex",
                      values="masa_kg", aggfunc="mean").round(2)
print(tabla.to_string())
islas = (p.groupby("island")
          .agg(n_isla=("species", "size"),
               especies=("species", "nunique")).reset_index())
p = p.merge(islas, on="island", how="left")         # unir tablas
print(p[["island", "n_isla", "especies"]].drop_duplicates()
      .to_string(index=False))
print("sin sexo:", p.sex.isna().sum(), "→ tras quitar:",
      len(p.dropna(subset=["sex"])))
extremos = p.dropna(subset=["pico_ratio"]).sort_values("pico_ratio")
print(extremos.iloc[[0, -1]][["species", "pico_ratio"]]
      .round(2).to_string())

`e2_graficos.py`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

p = pd.read_csv("penguins.csv").dropna()
fig, ejes = plt.subplots(2, 2, figsize=(4.4, 3.4),
                         layout="constrained")
for e, g in p.groupby("species"):                 # histogramas
    ejes[0, 0].hist(g.flipper_length_mm, bins=15, alpha=0.6,
                    label=e)
ejes[0, 0].set_title("Aleta (mm)"); ejes[0, 0].legend(fontsize=5)
datos = [g.body_mass_g / 1000 for _, g in p.groupby("species")]
ejes[0, 1].boxplot(datos, tick_labels=sorted(p.species.unique()))
ejes[0, 1].set_title("Masa (kg)")
cuenta = p.groupby(["island", "species"]).size().unstack(fill_value=0)
cuenta.plot.bar(stacked=True, ax=ejes[1, 0], rot=0, legend=False)
ejes[1, 0].set_title("Pingüinos por isla"); ejes[1, 0].set_xlabel("")
ejes[1, 1].scatter(p.flipper_length_mm, p.body_mass_g / 1000, s=3,
                   c=p.sex.map({"male": "C0", "female": "C1"}))
ejes[1, 1].set_title("Masa frente a aleta (azul: machos)")
plt.show()
print("figura con", len(fig.axes), "ejes guardada")

`e3_sklearn.py`

In [ ]:
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold

p = pd.read_csv("penguins.csv").dropna()
y = p.body_mass_g
cv = KFold(5, shuffle=True, random_state=0)
simple = make_pipeline(StandardScaler(), LinearRegression())
prep = make_column_transformer(
    (OneHotEncoder(drop="first"), ["species", "sex"]),
    (StandardScaler(), ["flipper_length_mm", "bill_length_mm"]))
completo = make_pipeline(prep, LinearRegression())
for nombre, m, X in [
        ("solo la aleta", simple, p[["flipper_length_mm"]]),
        ("especie, sexo, aleta, pico", completo,
         p[["species", "sex", "flipper_length_mm",
            "bill_length_mm"]])]:
    eam = -cross_val_score(m, X, y, cv=cv,
                           scoring="neg_mean_absolute_error")
    print(f"{nombre:<27} error absoluto medio {eam.mean():.0f} g"
          f" ± {eam.std():.0f}")
completo.fit(p, y)                 # el pipeline elige sus columnas
for n, c in zip(completo[0].get_feature_names_out(),
                completo[-1].coef_):
    print(f"  {n.split('__')[1]:<22} {c:8.1f}")

## 1.4 · Matemáticas para la IA con Python

`c9_autodiff.py`

In [ ]:
import tensorflow as tf

x = tf.Variable(1.0)
with tf.GradientTape() as cinta:            # graba las operaciones
    y = x ** 3 - 2 * x
print("dy/dx en x=1:", cinta.gradient(y, x).numpy())
# regla de la cadena en una neurona: p = sigmoide(w * x + b)
w, b = tf.Variable(0.5), tf.Variable(-1.0)
xx, etiqueta = tf.constant(3.0), tf.constant(1.0)
with tf.GradientTape() as cinta:
    z = w * xx + b
    p = tf.sigmoid(z)
    perdida = -(etiqueta * tf.math.log(p)
                + (1 - etiqueta) * tf.math.log(1 - p))
dw, db = cinta.gradient(perdida, [w, b])
print(f"p = {p.numpy():.4f}, pérdida = {perdida.numpy():.4f}")
print(f"automático: dL/dw = {dw.numpy():.4f},"
      f" dL/db = {db.numpy():.4f}")
print(f"a mano: (p - y) x = {((p - etiqueta) * xx).numpy():.4f},"
      f" (p - y) = {(p - etiqueta).numpy():.4f}")

## 1.6 · Las bibliotecas de la IA en Python

`e4_tensorflow.py`

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(0)
p = pd.read_csv("penguins.csv").dropna()
cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm",
        "body_mass_g"]
especies = sorted(p.species.unique())
y = p.species.map({e: i for i, e in enumerate(especies)}).to_numpy()
X_e, X_p, y_e, y_p = train_test_split(p[cols].to_numpy(float), y,
                                      test_size=0.3, stratify=y,
                                      random_state=0)
norm = tf.keras.layers.Normalization()
norm.adapt(X_e)                         # aprende medias y escalas
red = tf.keras.Sequential([
    tf.keras.Input(shape=(4,)), norm,
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")])
red.compile(optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
h = red.fit(X_e, y_e, epochs=60, batch_size=16, verbose=0)
print("parámetros:", red.count_params())
acc = h.history["accuracy"]
print(f"acierto entrenamiento: época 1 {acc[0]:.3f},"
      f" época 60 {acc[-1]:.3f}")
print(f"acierto prueba: {red.evaluate(X_p, y_p, verbose=0)[1]:.3f}")
prob = red.predict(X_p[:2], verbose=0).astype(float)
print("probabilidades de dos pingüinos:", prob.round(3).tolist(),
      "| real:", [especies[i] for i in y_p[:2]])

## 1.7 · Caso práctico: el sexo de los pingüinos que faltan

`f1_sexo.py`

In [ ]:
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

p = pd.read_csv("penguins.csv").dropna(subset=["body_mass_g"])
num = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm",
       "body_mass_g"]
conocido = p[p.sex.notna()]
modelo = make_pipeline(make_column_transformer(
    (OneHotEncoder(), ["species"]), (StandardScaler(), num)),
    LogisticRegression(max_iter=1000))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
ref = conocido.sex.value_counts(normalize=True).max()
acc = cross_val_score(modelo, conocido, conocido.sex, cv=cv)
print(f"referencia (decir siempre la clase mayor): {ref:.3f}")
print(f"regresión logística: acierto {acc.mean():.3f}"
      f" ± {acc.std():.3f}")
modelo.fit(conocido, conocido.sex)
falta = p[p.sex.isna()]
prob = modelo.predict_proba(falta)[:, 1]     # probabilidad de macho
t = falta[["species", "body_mass_g"]].assign(
    p_macho=prob.round(2), previsto=modelo.predict(falta))
print(t.to_string())

**Prueba.** En `a3_busqueda.py`, añade un obstáculo más al laberinto. ¿Cuántos estados exploran ahora la búsqueda en anchura y A*?

**Ejercicio.** En `f1_sexo.py`, quita la especie de las variables. ¿Cuánto baja el acierto?